# 01c -- dim_fantasy_teams Seed

**Purpose**: Build `dim_fantasy_teams` from the league's Google Sheet owner registry.
The sheet is the single source of truth for team names and
manager contacts.

**Key**: `team_key` (matches Google Sheet "Team ID" column: A01-A14, B01-B14)

**Source**: Google Sheet (Restricted), read through the Sheets API as a service
account: the key file's path is `GOOGLE_SHEETS_KEY_FILE` in `.env`, and the sheet
is shared with the service account as Viewer

**Three leagues, one kept**: the workbook lists three leagues' teams. The League
Associations tab (`CFG.league_assoc_sheet_gid`) maps each `League Name` to its
Fantrax `League ID`; 01c keeps the team rows of the League Name whose id is
`CFG.fantrax_league_id` and skips the rest (#724).

**Divisions → Conferences**:
- Riddell → A (A01–A14)
- Wilson  → B (B01–B14)

**Cap columns**: this table only carries the two cap facts that are true
independent of the roster -- `original_cap` (`CFG.total_cap`, never changes)
and `reinvestment_cap` (in-season bonus cap charges, 0 at seed time).
Everything roster-derived (active roster salary, dead money, remaining cap)
used to be an ETL rollup cached here (02e); it's now computed live instead
-- from `Fact_FantasyTeams` via the `_Measures.tmdl` DAX measures in Power
BI, and from the published `fact_team_cap.parquet` (built by `02h`) in
the bot. One formula, not a third cached copy.

**Outputs**:
- `data/dim_fantasy_teams.parquet`

In [ ]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import date
from dataclasses import dataclass, field


## 1 -- Load from Google Sheet

Re-run this notebook any time team names or manager emails change in the sheet.

In [ ]:
# -- Pull team registry from the Google Sheet --------------------------------
# The workbook is Restricted: read as the service account named by the key file
# in GOOGLE_SHEETS_KEY_FILE (.env), read-only scope. Only the columns below leave
# the sheet -- no email column (owner PII stays out of this repo).
# The workbook holds three leagues; League Associations maps each League Name to
# its Fantrax league id, so 01c can keep the football league's rows (#724).
_SHEET_COLS = ["Team ID", "Team Name", "Fantrax-TeamId", "Division", "League Name"]
_LEAGUE_COLS = ["Sport", "League Name", "League ID"]
print(f"Loading team sheet: {CFG.team_sheet_id} (gid {CFG.team_sheet_gid}, "
      f"leagues gid {CFG.league_assoc_sheet_gid})")
try:
    teams_raw = etl.read_sheet_tab(CFG.team_sheet_id, CFG.team_sheet_gid,
                                   columns=_SHEET_COLS)
    leagues_raw = etl.read_sheet_tab(CFG.team_sheet_id, CFG.league_assoc_sheet_gid,
                                     columns=_LEAGUE_COLS, text=["League ID"])
except etl.SheetsAuthError as e:
    raise SystemExit(str(e)) from None
except Exception as e:
    raise RuntimeError(
        f"Failed to load team sheet. Confirm it is shared with the service account as Viewer.\n{e}"
    ) from e

print(f"Rows loaded : {len(teams_raw)} teams, {len(leagues_raw)} league rows")
print(f"Columns     : {list(teams_raw.columns)} / {list(leagues_raw.columns)}")

## 2 -- Build dim_fantasy_teams

In [ ]:
# -- Column map: sheet -> dim_fantasy_teams -------------------------------------------
# Sheet columns: Division, Team Name, Team ID, Fantrax-TeamId, Roster
# Manager emails are owner PII: they are kept in a commissioners-only sheet and never
# in this one. This repo and its parquet are public (owner PII lives only in
# Supabase shared.owner, grill #87).
# Fantrax-TeamId is the league's authoritative teamId<->team_key identity
# (ADR-0005 locked column). Carrying it here lights up the ADR-0004 ledger join
# (teamId -> team_key in 02d) directly, retiring the name-match heuristic crosswalk.
_COL_MAP = {
    "Team ID":              "team_key",
    "Team Name":            "team_name",
    "Fantrax-TeamId":       "fantrax_team_id",
    "Division":             "division",
}

# Division -> conference letter (Riddell = A, Wilson = B)
_DIV_CONF = {"Riddell": "A", "Wilson": "B"}


def _text(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().replace("", pd.NA)


def keep_league_rows(teams: pd.DataFrame, leagues: pd.DataFrame, league_id: str) -> pd.DataFrame:
    # The workbook holds several leagues. The football league is the League Name whose
    # League ID (League Associations tab) is the configured Fantrax id -- never the
    # word in Sport. Stops print counts and column names only.
    missing = ([f"team tab {c}" for c in ["League Name"] if c not in teams.columns]
               + [f"League Associations {c}" for c in ["League Name", "League ID"]
                  if c not in leagues.columns])
    if missing:
        raise SystemExit(f"team-manifest is missing columns: {missing}")

    lg = pd.DataFrame({c: _text(leagues[c]) for c in ["League Name", "League ID"]}).dropna(how="all")
    match = lg[(lg["League ID"] == league_id).fillna(False).astype(bool)]
    if len(match) != 1:
        raise SystemExit(
            f"League Associations has {len(match)} rows for the configured Fantrax league id; "
            "it needs exactly one.")
    name = match["League Name"].iloc[0]
    if pd.isna(name):
        raise SystemExit(
            "League Associations League Name is blank on the row for the configured Fantrax league id.")

    tname = _text(teams["League Name"])
    blank = int(tname.isna().sum())
    unlisted = int((tname.notna() & ~tname.isin(set(lg["League Name"].dropna()))).sum())
    if blank or unlisted:
        raise SystemExit(
            f"team-manifest League Name is blank on {blank} rows and not listed in "
            f"League Associations on {unlisted}; every team needs a listed League Name.")

    keep = (tname == name).fillna(False).astype(bool).to_numpy()
    skipped = sorted(tname[~keep].value_counts().tolist())
    print(f"Kept {int(keep.sum())} rows of the configured league; "
          f"skipped {sum(skipped)} rows of {len(skipped)} other leagues (rows each: {skipped})")
    return teams.loc[keep].reset_index(drop=True)


def build_dim_fantasy_teams(raw: pd.DataFrame, cfg: LeagueConfig) -> pd.DataFrame:
    # Validate expected columns
    missing = [c for c in _COL_MAP if c not in raw.columns]
    if missing:
        raise ValueError(f"Sheet missing expected columns: {missing}")

    df = raw.rename(columns=_COL_MAP).copy()

    # Derive conference from division
    df["conference"] = df["division"].map(_DIV_CONF)
    unmapped_divs = df[df["conference"].isna()]["division"].unique()
    if len(unmapped_divs):
        raise ValueError(f"Unknown divisions -- add to _DIV_CONF: {list(unmapped_divs)}")

    # Fantrax-TeamId must be present + unique (it's the ledger identity key).
    if df["fantrax_team_id"].isna().any():
        raise ValueError(
            f"Blank Fantrax-TeamId for: {list(df.loc[df['fantrax_team_id'].isna(), 'team_key'])}")
    if not df["fantrax_team_id"].is_unique:
        raise ValueError("Fantrax-TeamId values are not unique")

    # The manifest must hold exactly CFG.num_teams teams, each with one unique Team ID.
    if len(df) != cfg.num_teams:
        raise SystemExit(
            f"team-manifest has {len(df)} rows but the league has {cfg.num_teams} teams; "
            "fix the manifest, or LeagueConfig if the league size changed.")
    if df["team_key"].isna().any() or not df["team_key"].is_unique:
        raise SystemExit(
            f"team-manifest Team ID is blank on {int(df['team_key'].isna().sum())} rows "
            f"and repeated on {int(df['team_key'].dropna().duplicated().sum())}; "
            "every team needs one unique Team ID.")

    # -- Cap state columns ---------------------------------------------------
    # original_cap/reinvestment_cap are the only true team-level cap facts --
    # everything roster-derived (active salary, dead money, remaining cap) is
    # computed live from fact_fantasy_teams instead of cached here (matches
    # the Power BI measures in _Measures.tmdl; the bot reads the
    # published fact_team_cap table, built by 02h). 02e no longer rolls anything up into this
    # table.
    df["original_cap"]      = cfg.total_cap
    df["reinvestment_cap"]  = 0

    # -- Final column order -----------------------------------------------
    cols = [
        "team_key", "team_name", "fantrax_team_id",
        "conference", "division",
        "original_cap", "reinvestment_cap",
    ]
    return df[cols].reset_index(drop=True)


dim_fantasy_teams = build_dim_fantasy_teams(
    keep_league_rows(teams_raw, leagues_raw, CFG.fantrax_league_id), CFG)

out_path = DATA / "dim_fantasy_teams.parquet"
etl.write_table("dim_fantasy_teams", dim_fantasy_teams)
print(f"dim_fantasy_teams: {len(dim_fantasy_teams)} teams -> {out_path}")
dim_fantasy_teams

## 3 -- Validation

In [ ]:
df = etl.read_table("dim_fantasy_teams")
print(f"dim_fantasy_teams: {len(df)} rows, {len(df.columns)} columns")
print()

# Team count by conference / division
summary = df.groupby(["conference", "division"]).agg(
    teams=("team_key", "count"),
).reset_index()
print("Teams by conference:")
print(summary.to_string(index=False))
print()

# Full roster display
display_cols = ["team_key", "team_name", "conference", "division"]
print("Full team roster:")
print(df[display_cols].to_string(index=False))